# Exploração de Resultados do Grid Search — Melhor Combinação por Modelo

Este notebook utiliza os resultados salvos em `RUN_DIR` do grid search para:
1. Carregar e pré-processar os datasets (`estabel`, `lav_temp`, `pecu`).
2. Re-treinar cada modelo com a **melhor combinação de hiperparâmetros** encontrada.
3. Calcular métricas clássicas e PU (Seção 6).
4. Plotar curvas ROC/PR e histogramas de scores.
5. Comparar desempenho individual vs. combinado.

## Seção 1 — Imports e Configurações Globais

In [ ]:
from pathlib import Path
import json
import math
import time
import hashlib
import itertools
import warnings


import numpy as np
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    roc_curve, precision_recall_curve, ndcg_score
)
from scipy.stats import spearmanr



from pyod.models.iforest import IForest
from pyod.models.hbos import HBOS
from pyod.models.ecod import ECOD
from pyod.models.copod import COPOD
from pyod.models.lof import LOF
from pyod.models.inne import INNE
from pyod.models.loda import LODA
from pyod.models.cblof import CBLOF
from pyod.models.auto_encoder import AutoEncoder


try:
    import torch
    AE_DEVICE = (
        "cuda" if torch.cuda.is_available()
        else "mps" if torch.backends.mps.is_available()
        else "cpu"
    )
except Exception:
    AE_DEVICE = "cpu"


import sys
# Garante que o scripts/ do projeto fique no path independentemente do cwd
try:
    _scripts_dir = Path(__file__).resolve().parent / "scripts"
except NameError:
    _scripts_dir = Path.cwd() / "scripts"

if not (_scripts_dir / "encoder.py").exists():
    _scripts_dir = (Path.cwd() / "04 - AprendizadoDeMaquina" / "scripts")

if str(_scripts_dir) not in sys.path:
    sys.path.append(str(_scripts_dir))


from encoder import AnomalyDetectionEncoder


warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110


pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", 200)


print(f"AE_DEVICE: {AE_DEVICE}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Configuração do experimento
# ─────────────────────────────────────────────────────────────
RUN_DIR = Path("../data/resultados/gs_ap_combined_100k_20260518_231005")

KEY_COLS = ["V010100", "NUM_QUADRA", "NUM_FACE", "V010800"]
CONTAMINATION = 0.0687
SAMPLE_SIZE = 1_000
RANDOM_STATE = 42

LABELS_PATH = Path("../data/experimentos/ground_truth.csv")
DATA_DIR = Path("../data/experimentos/abordagem1")
DATASET_FILES = {
    "estabel": "df_estabel_final.parquet",
    "lav_temp": "df_lav_temp_final.parquet",
    "pecu": "df_pecu_final.parquet",
}

if not RUN_DIR.exists():
    raise FileNotFoundError(f"RUN_DIR não encontrado: {RUN_DIR.resolve()}")

print("RUN_DIR:", RUN_DIR.resolve())
for p in sorted(RUN_DIR.glob("*")):
    print("-", p.name)

## Seção 2 — Especificação dos Modelos (MODEL_SPECS)

Replicamos os modelos e grades do notebook `02_grid_search.ipynb`. Esses specs são usados para instanciar cada modelo com a melhor combinação encontrada no grid search.

In [ ]:
MODEL_SPECS = {
    "IForest": {
        "cls": IForest,
        "fixed": {"random_state": RANDOM_STATE, "n_jobs": -1},
        "grid": {
            "n_estimators": [100, 200, 400],
            "max_samples": [256, 512, 1024],
            "max_features": [0.1, 0.4, 0.6, 0.8, 1.0],
            "bootstrap": [False, True],
        },
        "family": "Ensemble",
    },
    "LODA": {
        "cls": LODA,
        "fixed": {},
        "grid": {
            "n_bins": [20, 50, 100, 200, 400],
            "n_random_cuts": [20, 50, 100, 200],
        },
        "family": "Ensemble",
    },
    "HBOS": {
        "cls": HBOS,
        "fixed": {},
        "grid": {
            "n_bins": [5, 10, 15, 20],
            "alpha": [0.05, 0.1, 0.3, 0.5, 0.8, 1.0],
            "tol": [0.3, 0.5, 0.6, 0.7],
        },
        "family": "Histograma",
    },
    "CBLOF": {
        "cls": CBLOF,
        "fixed": {"random_state": RANDOM_STATE},
        "grid": {
            "n_clusters": [2, 4, 6, 8, 12],
            "alpha": [0.3, 0.7, 0.9, 1],
            "beta": [3, 5],
            "use_weights": [False],
        },
        "family": "Proximity",
    },
    "INNE": {
        "cls": INNE,
        "fixed": {"random_state": RANDOM_STATE},
        "grid": {
            "n_estimators": [50, 100, 200, 400, 800],
            "max_samples": [256, 512, 1024],
        },
        "family": "Ensemble",
    },
    "LOF": {
        "cls": LOF,
        "fixed": {"n_jobs": -1},
        "grid": {
            "n_neighbors": [10, 20, 35, 50, 75],
            "leaf_size": [10, 20, 30, 50],
            "p": [1],
        },
        "family": "Proximity",
    },
    "ECOD": {
        "cls": ECOD,
        "fixed": {"n_jobs": -1},
        "grid": {},
        "family": "Probabilistic",
    },
    "COPOD": {
        "cls": COPOD,
        "fixed": {"n_jobs": -1},
        "grid": {},
        "family": "Probabilistic",
    },
    "AutoEncoder": {
        "cls": AutoEncoder,
        "fixed": {"random_state": RANDOM_STATE, "device": AE_DEVICE},
        "grid": {
            "hidden_neuron_list": [[64, 32, 64], [64, 32, 32, 64], [64, 32, 32, 32, 64]],
            "epoch_num": [20, 40, 60, 80],
            "batch_size": [32, 64, 128],
            "lr": [1e-4, 1e-3, 1e-2, 1e-1],
        },
        "family": "Neural",
    },
}

def make_param_combinations(param_grid: dict):
    keys = list(param_grid.keys())
    values = list(param_grid.values())
    for combo in itertools.product(*values):
        yield dict(zip(keys, combo))

def combo_id_from_params(params: dict) -> str:
    param_json = json.dumps(params, sort_keys=True, default=str)
    return hashlib.md5(param_json.encode("utf-8")).hexdigest()[:12]

print("MODEL_SPECS criados com os modelos:")
for name, spec in MODEL_SPECS.items():
    n = math.prod([len(v) for v in spec["grid"].values()]) if spec["grid"] else 1
    print(f"  [{spec['family']:15s}] {name:<12s} — {n} combinação(ões)")

## Seção 3 — Carregamento do Ground Truth e Pré-processamento dos Datasets

In [ ]:
def safe_ap(y_true, scores):
    y = np.asarray(y_true)
    s = np.asarray(scores)
    if len(y) == 0 or np.unique(y).size < 2:
        return float("nan")
    return float(average_precision_score(y, s))

def safe_auc(y_true, scores):
    y = np.asarray(y_true)
    s = np.asarray(scores)
    if len(y) == 0 or np.unique(y).size < 2:
        return float("nan")
    return float(roc_auc_score(y, s))

def preprocessar_dataset(nome: str, parquet_path: Path, labels_df: pd.DataFrame) -> dict:
    df = pd.read_parquet(parquet_path)

    for col in KEY_COLS:
        if col not in df.columns:
            raise KeyError(f"Coluna {col} não encontrada no dataset {nome}.")
        df[col] = df[col].astype(str)

    df = df.merge(labels_df[KEY_COLS + ["is_anomaly"]], on=KEY_COLS, how="left")
    df["is_anomaly"] = df["is_anomaly"].fillna(0).astype(int)

    n_target = min(SAMPLE_SIZE, len(df))
    if n_target < len(df):
        sampled = []
        for label, grp in df.groupby("is_anomaly"):
            n_grp = max(1, round(n_target * len(grp) / len(df)))
            n_grp = min(n_grp, len(grp))
            sampled.append(grp.sample(n=n_grp, random_state=RANDOM_STATE))
        df = pd.concat(sampled, axis=0).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

    y = df["is_anomaly"].values
    if np.unique(y).size < 2:
        raise ValueError(f"Dataset {nome} ficou com apenas uma classe após amostragem.")

    key_df = df[KEY_COLS].copy()
    X = df.drop(columns=KEY_COLS + ["is_anomaly"], errors="ignore")

    X_train, X_test, y_train, y_test, key_train, key_test = train_test_split(
        X, y, key_df,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    encoder = AnomalyDetectionEncoder(key_cols=[])
    encoder.fit(X_train)
    X_train_enc = encoder.transform(X_train)
    X_test_enc = encoder.transform(X_test)

    imputer = SimpleImputer(strategy="constant", fill_value=0)
    X_train_enc = imputer.fit_transform(X_train_enc)
    X_test_enc = imputer.transform(X_test_enc)

    feature_names = encoder.get_feature_info()["feature"].tolist() if hasattr(encoder, "get_feature_info") else list(X_train.columns)

    return {
        "X_train": X_train_enc,
        "X_test": X_test_enc,
        "y_train": y_train,
        "y_test": pd.Series(y_test, name="is_anomaly"),
        "key_test": key_test.reset_index(drop=True),
        "feature_names": feature_names,
        "encoder": encoder,
        "imputer": imputer,
    }


if not LABELS_PATH.exists():
    raise FileNotFoundError(f"Arquivo de labels não encontrado: {LABELS_PATH}")

labels_df = pd.read_csv(LABELS_PATH)
required_cols = KEY_COLS + ["is_anomaly"]
missing = [c for c in required_cols if c not in labels_df.columns]
if missing:
    raise ValueError(f"Colunas ausentes em labels: {missing}")

for c in KEY_COLS:
    labels_df[c] = labels_df[c].astype(str)
labels_df["is_anomaly"] = labels_df["is_anomaly"].astype(int)

datasets = {}
for ds_name, filename in DATASET_FILES.items():
    parquet_path = DATA_DIR / filename
    if not parquet_path.exists():
        print(f"[SKIP] Dataset ausente: {ds_name}")
        continue
    t0 = time.time()
    datasets[ds_name] = preprocessar_dataset(ds_name, parquet_path, labels_df)
    print(f"[OK] {ds_name}: train={datasets[ds_name]['X_train'].shape}, test={datasets[ds_name]['X_test'].shape}, tempo={time.time()-t0:.1f}s")

if not datasets:
    raise RuntimeError("Nenhum dataset válido foi pré-processado.")

## Seção 4 — Carregamento da Melhor Combinação por Modelo

Lemos `best_by_model.csv` do grid search e identificamos, para cada modelo, qual `combo_id` obteve o melhor `objective_ap`.

In [ ]:
best_by_model = pd.read_csv(RUN_DIR / "best_by_model.csv")
best_by_model = best_by_model[best_by_model["status"].fillna("") == "ok"].copy()

# Mantém apenas modelos presentes em MODEL_SPECS
best_by_model = best_by_model[best_by_model["model"].isin(MODEL_SPECS.keys())].copy()

# Melhor combinação por modelo (maior objective_ap)
best_by_model = best_by_model.sort_values("objective_ap", ascending=False).groupby("model", as_index=False).first()

# ECOD e COPOD têm grid vazio; se não estiverem no best_by_model, forçamos inclusão
for prob_model in ["ECOD", "COPOD"]:
    if prob_model not in best_by_model["model"].values and prob_model in MODEL_SPECS:
        empty_params = {}
        combo = combo_id_from_params(empty_params)
        best_by_model = pd.concat([
            best_by_model,
            pd.DataFrame([{
                "model": prob_model,
                "combo_id": combo,
                "params_json": json.dumps(empty_params),
                "objective_ap": float("nan"),
                "ap_combined": float("nan"),
                "ap_individual_mean": float("nan"),
                "status": "ok",
            }])
        ], ignore_index=True)

print("Melhor combinação por modelo:")
print(best_by_model[["model", "combo_id", "objective_ap", "ap_combined", "ap_individual_mean"]].to_string(index=False))

## Seção 5 — Treinamento e Scoring com as Melhores Combinações

Para cada modelo listado em `best_by_model`, instanciamos a classe correspondente com os hiperparâmetros da melhor combinação, treinamos em cada dataset e armazenamos scores, predições, tempo e o modelo ajustado.

In [ ]:
def treinar_e_avaliar(model_name: str, model, X_train, X_test):
    t0 = time.time()
    model.fit(X_train)
    scores = model.decision_function(X_test)
    preds = model.predict(X_test)
    elapsed = time.time() - t0
    return scores, preds, elapsed, model


all_results = {}

for _, row in best_by_model.iterrows():
    model_name = row["model"]
    combo_id = row["combo_id"]
    spec = MODEL_SPECS[model_name]

    # Recupera os parâmetros que geram o combo_id
    params = None
    for candidate in make_param_combinations(spec["grid"]):
        if combo_id_from_params(candidate) == combo_id:
            params = candidate
            break

    if params is None:
        print("[AVISO] Não foi possível recuperar params para " + model_name + " / " + combo_id + "; tentando ler params_json de best_by_model.")
        try:
            params = json.loads(row["params_json"])
        except Exception as e:
            print("  -> falha: " + str(e) + "; pulando " + model_name)
            continue

    print("[" + model_name + "] combo=" + combo_id + " | params=" + str(params))
    all_results[model_name] = {}

    for ds_name, ds in datasets.items():
        model = spec["cls"](contamination=CONTAMINATION, **spec["fixed"], **params)
        scores, preds, elapsed, fitted = treinar_e_avaliar(model_name, model, ds["X_train"], ds["X_test"])
        all_results[model_name][ds_name] = {
            "scores": scores,
            "preds": preds,
            "time": elapsed,
            "fitted_model": fitted,
            "combo_id": combo_id,
            "params": params,
        }
        n_flagged = (preds == 1).sum()
        print("  " + ds_name + " tempo=" + f"{elapsed:6.1f}" + "s flagged=" + f"{n_flagged:5d}" + " (" + f"{n_flagged/len(preds):.2%}" + ")")

print("Treinamento das melhores combinações concluído.")
print("Modelos treinados: " + str(list(all_results.keys())))


In [ ]:
def treinar_e_avaliar(model_name: str, model, X_train, X_test):
    t0 = time.time()
    model.fit(X_train)
    scores = model.decision_function(X_test)
    preds = model.predict(X_test)
    elapsed = time.time() - t0
    return scores, preds, elapsed, model


all_results = {}

for _, row in best_by_model.iterrows():
    model_name = row["model"]
    combo_id = row["combo_id"]
    spec = MODEL_SPECS[model_name]

    # Recupera os parâmetros que geram o combo_id
    params = None
    for candidate in make_param_combinations(spec["grid"]):
        if combo_id_from_params(candidate) == combo_id:
            params = candidate
            break

    if params is None:
        print("[AVISO] Não foi possível recuperar params para " + model_name + " / " + combo_id + "; tentando ler params_json de best_by_model.")
        try:
            params = json.loads(row["params_json"])
        except Exception as e:
            print("  -> falha: " + str(e) + "; pulando " + model_name)
            continue

    print("[" + model_name + "] combo=" + combo_id + " | params=" + str(params))
    all_results[model_name] = {}

    for ds_name, ds in datasets.items():
        model = spec["cls"](contamination=CONTAMINATION, **spec["fixed"], **params)
        scores, preds, elapsed, fitted = treinar_e_avaliar(model_name, model, ds["X_train"], ds["X_test"])
        all_results[model_name][ds_name] = {
            "scores": scores,
            "preds": preds,
            "time": elapsed,
            "fitted_model": fitted,
            "combo_id": combo_id,
            "params": params,
        }
        n_flagged = (preds == 1).sum()
        print("  " + ds_name + " tempo=" + f"{elapsed:6.1f}" + "s flagged=" + f"{n_flagged:5d}" + " (" + f"{n_flagged/len(preds):.2%}" + ")")

print("Treinamento das melhores combinações concluído.")
print("Modelos treinados: " + str(list(all_results.keys())))

In [ ]:

def precision_at_k(y_true, scores, k: int) -> float:
    top_k_idx = np.argsort(scores)[::-1][:k]
    return float(np.asarray(y_true)[top_k_idx].mean())

def recall_at_k(y_true, scores, k: int) -> float:
    y = np.asarray(y_true)
    n_pos = int(y.sum())
    if n_pos == 0:
        return 0.0
    top_k_idx = np.argsort(scores)[::-1][:k]
    return float(y[top_k_idx].sum() / n_pos)

def mean_normalized_rank(y_true, scores) -> float:
    y = np.asarray(y_true)
    n = len(y)
    order = np.argsort(scores)[::-1]
    ranks = np.empty(n, dtype=float)
    ranks[order] = np.arange(n)
    pos_ranks = ranks[y == 1]
    if len(pos_ranks) == 0:
        return 0.5
    return float(pos_ranks.mean() / (n - 1))

def ndcg_at_k(y_true, scores, k: int) -> float:
    y = np.asarray(y_true, dtype=float)
    s = np.asarray(scores, dtype=float)
    return float(ndcg_score(y.reshape(1, -1), s.reshape(1, -1), k=k))

def calcular_metricas(y_true, scores, contamination: float = CONTAMINATION) -> dict:
    y = np.asarray(y_true)
    n_pos = max(1, int(y.sum()))
    k_cont = max(1, math.floor(len(y) * contamination))

    threshold = np.percentile(scores, (1 - contamination) * 100)
    preds_at_cont = (scores >= threshold).astype(int)
    tp = int(((preds_at_cont == 1) & (y == 1)).sum())
    fp = int(((preds_at_cont == 1) & (y == 0)).sum())
    fn = int(((preds_at_cont == 0) & (y == 1)).sum())
    denom = 2 * tp + fp + fn
    f1 = (2 * tp / denom) if denom > 0 else 0.0

    k_pu = n_pos
    k_pu2 = 2 * n_pos

    return {
        "avg_precision": average_precision_score(y, scores),
        "roc_auc": roc_auc_score(y, scores),
        "precision_at_k": precision_at_k(y, scores, k_cont),
        "f1_at_cont": f1,
        "k": k_cont,
        "recall_at_k": recall_at_k(y, scores, k_pu),
        "recall_at_2k": recall_at_k(y, scores, k_pu2),
        "mean_norm_rank": mean_normalized_rank(y, scores),
        "ndcg_at_k": ndcg_at_k(y, scores, k_pu),
    }


def build_combined_df(per_ds_score_dfs: list[pd.DataFrame]) -> pd.DataFrame:
    if not per_ds_score_dfs:
        return pd.DataFrame()
    merged = per_ds_score_dfs[0].copy()
    for part in per_ds_score_dfs[1:]:
        merged = merged.merge(part, on=KEY_COLS, how="outer")
    score_cols = [c for c in merged.columns if c.startswith("score_")]
    y_cols = [c for c in merged.columns if c.startswith("y_true_")]
    merged["combined_score"] = merged[score_cols].max(axis=1, skipna=True)
    if y_cols:
        merged["y_true"] = merged[y_cols].max(axis=1, skipna=True).fillna(0).astype(int)
    else:
        merged["y_true"] = 0
    out_cols = KEY_COLS + ["combined_score", "y_true"]
    return merged[out_cols].dropna(subset=["combined_score"]).reset_index(drop=True)


metrics_rows = []
for model_name, ds_results in all_results.items():
    family = MODEL_SPECS[model_name]["family"]
    for ds_name, res in ds_results.items():
        y_test = datasets[ds_name]["y_test"]
        m = calcular_metricas(y_test, res["scores"])
        metrics_rows.append({
            "dataset": ds_name,
            "model": model_name,
            "family": family,
            "avg_precision": m["avg_precision"],
            "roc_auc": m["roc_auc"],
            "precision_at_k": m["precision_at_k"],
            "f1_at_cont": m["f1_at_cont"],
            "recall_at_k": m["recall_at_k"],
            "recall_at_2k": m["recall_at_2k"],
            "mean_norm_rank": m["mean_norm_rank"],
            "ndcg_at_k": m["ndcg_at_k"],
            "time_s": res["time"],
        })

ranking_df = pd.DataFrame(metrics_rows).sort_values(["dataset", "avg_precision"], ascending=[True, False])

_cols_show = ["dataset", "model", "family", "avg_precision", "roc_auc",
              "precision_at_k", "f1_at_cont", "recall_at_k", "recall_at_2k",
              "mean_norm_rank", "ndcg_at_k", "time_s"]

combined_rows = []
combined_dfs = {}

for model_name in all_results.keys():
    per_ds = []
    for ds_name, res in all_results[model_name].items():
        ds = datasets[ds_name]
        df_out = ds["key_test"].copy()
        df_out[f"score_{ds_name}"] = res["scores"]
        df_out[f"y_true_{ds_name}"] = ds["y_test"].values
        per_ds.append(df_out)

    combined_df = build_combined_df(per_ds)
    combined_dfs[model_name] = combined_df
    if combined_df.empty or combined_df["y_true"].nunique() < 2:
        continue
    m = calcular_metricas(combined_df["y_true"].values, combined_df["combined_score"].values)
    combined_rows.append({
        "dataset": "combined",
        "model": model_name,
        "family": MODEL_SPECS[model_name]["family"],
        "avg_precision": m["avg_precision"],
        "roc_auc": m["roc_auc"],
        "precision_at_k": m["precision_at_k"],
        "f1_at_cont": m["f1_at_cont"],
        "recall_at_k": m["recall_at_k"],
        "recall_at_2k": m["recall_at_2k"],
        "mean_norm_rank": m["mean_norm_rank"],
        "ndcg_at_k": m["ndcg_at_k"],
        "time_s": float("nan"),
    })

if combined_rows:
    ranking_df = pd.concat([ranking_df, pd.DataFrame(combined_rows)], ignore_index=True)

print("Ranking por Average Precision (com métricas PU e avaliação combinada):\n")
print(ranking_df[_cols_show].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
def _plot_curvas(ds_name, ds_results, y_test):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    palette = sns.color_palette("tab10", n_colors=len(ds_results))

    for (model_name, res), color in zip(ds_results.items(), palette):
        scores = res["scores"]
        fpr, tpr, _ = roc_curve(y_test, scores)
        auc_val = roc_auc_score(y_test, scores)
        axes[0].plot(fpr, tpr, label=f"{model_name} (AUC={auc_val:.3f})", color=color)
        prec, rec, _ = precision_recall_curve(y_test, scores)
        ap_val = average_precision_score(y_test, scores)
        axes[1].plot(rec, prec, label=f"{model_name} (AP={ap_val:.3f})", color=color)

    pos_rate = np.asarray(y_test).mean()
    axes[0].plot([0, 1], [0, 1], "k--", lw=0.8, label="Random")
    axes[1].axhline(pos_rate, color="k", ls="--", lw=0.8, label=f"Random (AP={pos_rate:.3f})")

    axes[0].set(title=f"Curva ROC — {ds_name}", xlabel="FPR", ylabel="TPR")
    axes[1].set(title=f"Curva PR — {ds_name}", xlabel="Recall", ylabel="Precision")
    for ax in axes:
        ax.legend(fontsize=8)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()


def _plot_score_hist(ds_name, ds_results, y_test):
    n_models = len(ds_results)
    cols = min(3, n_models)
    rows = math.ceil(n_models / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.5 * rows))
    axes = np.array(axes).flatten()

    for idx, (model_name, res) in enumerate(ds_results.items()):
        scores = res["scores"]
        ax = axes[idx]
        for label, color, name in [(0, "steelblue", "Normal"), (1, "tomato", "Anomalia")]:
            mask = np.asarray(y_test) == label
            ax.hist(scores[mask], bins=40, alpha=0.6, color=color,
                    label=f"{name} (n={mask.sum()})", density=True)
        ax.set_title(f"{model_name}", fontsize=10)
        ax.set_xlabel("Score")
        ax.legend(fontsize=8)

    for ax in axes[n_models:]:
        ax.set_visible(False)

    plt.suptitle(f"Distribuição de Scores por Classe — {ds_name}", fontsize=12)
    plt.tight_layout()
    plt.show()


for model_name, ds_results in all_results.items():
    for ds_name, res in ds_results.items():
        y_test = datasets[ds_name]["y_test"]
        _plot_curvas(ds_name, {model_name: res}, y_test)
        _plot_score_hist(ds_name, {model_name: res}, y_test)

if combined_dfs:
    for model_name, df in combined_dfs.items():
        valid = df.dropna(subset=["combined_score"])
        if valid["y_true"].nunique() < 2:
            continue
        _plot_curvas("combined", {model_name: {"scores": valid["combined_score"].values}}, valid["y_true"].values)
        _plot_score_hist("combined", {model_name: {"scores": valid["combined_score"].values}}, valid["y_true"].values)

In [ ]:

individual_best = (
    ranking_df[ranking_df["dataset"] != "combined"]
    .groupby("model")["avg_precision"].max().reset_index()
    .rename(columns={"avg_precision": "AP_individual_best"})
)
combined_ap = (
    ranking_df[ranking_df["dataset"] == "combined"][["model", "avg_precision"]]
    .rename(columns={"avg_precision": "AP_combined"})
)
comp = combined_ap.merge(individual_best, on="model")

if not comp.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(comp))
    width = 0.35
    bars1 = ax.bar(x - width/2, comp["AP_individual_best"], width, label="Melhor individual", color="steelblue")
    bars2 = ax.bar(x + width/2, comp["AP_combined"], width, label="Combinado (max)", color="tomato")
    ax.set_xticks(x)
    ax.set_xticklabels(comp["model"], rotation=20, ha="right")
    ax.set_ylabel("Average Precision")
    ax.set_title("AP: Melhor Dataset Individual vs. Score Combinado")
    ax.legend()
    ax.set_ylim(0, 1)
    for bar in list(bars1) + list(bars2):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=7)
    plt.tight_layout()
    plt.show()

## Seção 7 — Análise de Concordância e Explicabilidade

In [ ]:

concordance_summary = {}

for ds_name in datasets.keys():
    ds_results = {model: res[ds_name] for model, res in all_results.items() if ds_name in res}
    if not ds_results:
        continue

    y_test = datasets[ds_name]["y_test"]
    key_test = datasets[ds_name]["key_test"]
    model_names = list(ds_results.keys())
    n = len(model_names)

    scores_matrix = np.column_stack([ds_results[m]["scores"] for m in model_names])
    corr_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            corr_matrix[i, j] = spearmanr(scores_matrix[:, i], scores_matrix[:, j]).statistic

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(
        pd.DataFrame(corr_matrix, index=model_names, columns=model_names),
        annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
        ax=axes[0], square=True
    )
    axes[0].set_title(f"Correlação de Spearman (scores) — {ds_name}")

    preds_matrix = np.column_stack([ds_results[m]["preds"] for m in model_names])
    consensus = preds_matrix.sum(axis=1)

    consensus_df = key_test.copy()
    consensus_df["consensus_count"] = consensus
    consensus_df["is_anomaly"] = y_test.values
    concordance_summary[ds_name] = consensus_df

    max_count = int(consensus.max()) + 1
    labels_order = list(range(max_count))
    counts = pd.Series(consensus).value_counts().reindex(labels_order, fill_value=0)
    counts.plot(kind="bar", ax=axes[1], color="steelblue", rot=0)
    axes[1].set_xlabel("Nº de modelos que sinalizaram anomalia")
    axes[1].set_ylabel("Nº de registros")
    axes[1].set_title(f"Histograma de Consenso — {ds_name}")
    for bar in axes[1].patches:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     str(int(bar.get_height())), ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.show()

    print(f"\nTop-20 registros por consenso — {ds_name}:")
    top20 = consensus_df.sort_values("consensus_count", ascending=False).head(20).reset_index(drop=True)
    print(top20.to_string())